## 6. Model Evaluation

Comprehensive evaluation of all trained models on the held-out test set.  
Primary metric: **AUC-ROC** (as specified by Mastercard MDQ).  
Secondary: precision, recall, F1, accuracy, confusion matrix, overfitting analysis.


### 6.1 Collect predictions from all models

In [ ]:
models_to_eval = {
    'Logistic Regression': (lr_pipeline,  X_test,     y_test),
    'Random Forest':       (rf_model,     X_test,     y_test),
    'XGBoost (tuned)':     (best_xgb,     X_test,     y_test),
    'CatBoost':            (cat_model,    X_test,     y_test),
    f'{FINAL_MODEL_NAME} (final)': (FINAL_MODEL, X_test_sel, y_test),
}

eval_summary = []

for name, (model, X_ev, y_ev) in models_to_eval.items():
    prob  = model.predict_proba(X_ev)[:, 1]
    pred  = (prob >= 0.5).astype(int)
    auc   = roc_auc_score(y_ev, prob)
    prec  = precision_score(y_ev, pred, zero_division=0)
    rec   = recall_score(y_ev, pred, zero_division=0)
    f1    = f1_score(y_ev, pred, zero_division=0)
    acc   = accuracy_score(y_ev, pred)
    eval_summary.append({
        'Model': name, 'AUC-ROC': round(auc, 4),
        'Precision': round(prec, 4), 'Recall': round(rec, 4),
        'F1': round(f1, 4), 'Accuracy': round(acc, 4)
    })

eval_df = pd.DataFrame(eval_summary).sort_values('AUC-ROC', ascending=False)
display(eval_df)

### 6.2 ROC curves — all models

In [ ]:
plt.figure(figsize=(10, 7))

for name, (model, X_ev, y_ev) in models_to_eval.items():
    prob = model.predict_proba(X_ev)[:, 1]
    fpr, tpr, _ = roc_curve(y_ev, prob)
    auc = roc_auc_score(y_ev, prob)
    plt.plot(fpr, tpr, label=f'{name}  (AUC={auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 6.3 Confusion matrix — final model (threshold = 0.5)

In [ ]:
prob_final = FINAL_MODEL.predict_proba(X_test_sel)[:, 1]
pred_final = (prob_final >= 0.5).astype(int)

cm = confusion_matrix(y_test, pred_final)
disp = ConfusionMatrixDisplay(cm, display_labels=['Consumer (0)', 'Business (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {FINAL_MODEL_NAME}')
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_final,
      target_names=['Consumer (0)', 'Business (1)']))

### 6.4 Overfitting analysis — train vs test AUC

In [ ]:
print('Overfitting check (Train AUC vs Test AUC):\n')
for name, (model, X_ev, y_ev) in models_to_eval.items():
    X_tr_ev = X_train[SELECTED_FEATURES] if 'final' in name.lower() else X_train
    prob_tr = model.predict_proba(X_tr_ev)[:, 1]
    prob_te = model.predict_proba(X_ev)[:, 1]
    auc_tr  = roc_auc_score(y_train, prob_tr)
    auc_te  = roc_auc_score(y_ev,    prob_te)
    gap     = auc_tr - auc_te
    flag    = '⚠️  overfitting' if gap > 0.05 else '✅'
    print(f'{name:35s}  Train: {auc_tr:.4f}  Test: {auc_te:.4f}  Gap: {gap:.4f}  {flag}')

### 6.5 CV results summary — stability check

In [ ]:
print('Cross-validation AUC summary (on training set):\n')
cv_rows = []
for name, scores in cv_results.items():
    cv_rows.append({
        'Model': name,
        'CV AUC mean': round(scores['auc'].mean(), 4),
        'CV AUC std':  round(scores['auc'].std(), 4),
        'Min fold AUC': round(scores['auc'].min(), 4),
        'Max fold AUC': round(scores['auc'].max(), 4),
    })
cv_df = pd.DataFrame(cv_rows).sort_values('CV AUC mean', ascending=False)
display(cv_df)

fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, scores) in enumerate(cv_results.items()):
    ax.scatter([name]*N_FOLDS, scores['auc'], alpha=0.7, s=60)
    ax.plot([i-0.3, i+0.3], [scores['auc'].mean()]*2, color='black', linewidth=2)
ax.set_ylabel('AUC-ROC')
ax.set_title('Cross-validation AUC per fold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 6.6 Threshold optimization

Default threshold = 0.5 may not be optimal for an imbalanced setting.  
We scan thresholds and pick the one maximizing **F1** on the test set.


In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)
f1_scores  = [f1_score(y_test, (prob_final >= t).astype(int), zero_division=0)
              for t in thresholds]

best_idx       = int(np.argmax(f1_scores))
BEST_THRESHOLD = thresholds[best_idx]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_scores)
plt.axvline(BEST_THRESHOLD, color='red', linestyle='--',
            label=f'Best threshold = {BEST_THRESHOLD:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 score')
plt.title('Threshold vs F1 — final model')
plt.legend()
plt.tight_layout()
plt.show()

pred_opt = (prob_final >= BEST_THRESHOLD).astype(int)
print(f'Optimal threshold: {BEST_THRESHOLD:.2f}')
print(classification_report(y_test, pred_opt,
      target_names=['Consumer (0)', 'Business (1)']))

### 6.7 Model comparison summary

In [ ]:
print('Final model ranking by AUC-ROC:')
display(eval_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if n == eval_df.iloc[0]['Model'] else '#3498db'
          for n in eval_df['Model']]
ax.barh(eval_df['Model'], eval_df['AUC-ROC'], color=colors)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('AUC-ROC')
ax.set_title('Model Comparison — Test AUC-ROC')
ax.invert_yaxis()
for bar, val in zip(ax.patches, eval_df['AUC-ROC']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center')
plt.tight_layout()
plt.show()